# Métricas — Contratos de Handoff

Agrega os `.jsonl` de `data/logs/` e calcula as quatro dimensões da seção 9 da proposta.

**Nada é calculado aqui**: o cálculo mora em [`handoff/measure.py`](../handoff/measure.py),
testado em `tests/test_measure.py`. Este notebook só agrega, faz estatística e plota — de
modo que a definição de cada métrica seja única e citável na metodologia.

Regras herdadas do desenho (ver `analysis/README.md`):

1. a unidade de análise é a **pergunta**, não a célula — seeds não geram variância neste modelo;
2. toda métrica de containment se compara ao **controle da mesma condição, pergunta e seed**;
3. `fate == "sem_efeito"` não é contenção;
4. RCCA/FESA só se leem contra a taxa de falso-positivo do auditor nas células limpas.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from handoff.measure import load_cells, to_records

# Troque para o experimento que quiser analisar -- cada modelo tem sua
# propria pasta, nunca misturada com os outros dois:
#   'gpt-5.6-luna'                   (experimento 1, OpenAI)
#   'claude-haiku-4-5-20251001'      (experimento 2, Anthropic)
#   'qwen3.5-9b-mlx'                 (experimento 3, SLM local/Ollama)
MODEL_SLUG = 'gpt-5.6-luna'

LOG_DIR = Path.cwd().parent / 'data' / MODEL_SLUG / 'logs'
CONDICOES = ['A', 'B', 'C', 'D']

cells = load_cells(LOG_DIR)
df = pd.DataFrame(to_records(cells))
controles = [c for c in cells if c.is_control]
print(f'modelo: {MODEL_SLUG}')
print(f'{len(cells)} celulas ({len(controles)} de controle) -> {len(df)} linhas injetadas')
df.head()

## 0. Integridade da matriz

Antes de qualquer média: a matriz está completa? Célula sem contrafactual limpo entra com
as métricas de containment em `None` de propósito — descartá-las em silêncio esconderia
buraco na matriz.

In [ ]:
print('celulas sem controle pareado:', int((~df.has_control).sum()))
print('respostas sem gold numerico (Correctness nao mensuravel):', int(df.final_correct.isna().sum()))

cobertura = (df.pivot_table(index=['injection_stage', 'error_type'],
                            columns='condition', values='question_id', aggfunc='nunique')
              .fillna(0).astype(int))
display(cobertura)

desbalanceada = cobertura[cobertura.nunique(axis=1) > 1]
if len(desbalanceada):
    print('ATENCAO: braços com n diferente entre condicoes -- a comparacao pareada abaixo',
          'usa so as perguntas presentes nas 4 condicoes')
    display(desbalanceada)

## 1. Seeds são réplicas independentes?

Premissa congelada em `CONTEXT.md` §11.4: com este modelo as seeds convergem, então tratá-las
como observações independentes estreitaria o IC artificialmente. **Confira nos seus próprios
dados** — e refaça sempre que o modelo ou o `reasoning_effort` mudar.

In [ ]:
seeds = sorted(df.seed.unique())
if len(seeds) > 1:
    piv = df.pivot_table(index=['question_id', 'condition', 'error_type', 'injection_stage'],
                         columns='seed', values='esr')
    piv = piv.dropna(subset=seeds[:2])
    disc = (piv[seeds[0]] != piv[seeds[1]]).mean()
    print(f'discordancia de ESR entre seed {seeds[0]} e {seeds[1]}: {disc:.1%} (n={len(piv)})')
    print('~0 => rode uma seed so; a replica nao adiciona informacao')
else:
    print(f'uma seed ({seeds[0]}): unidade de analise = pergunta, como planejado')

## 2. Agregação por pergunta e intervalos de confiança

`por_pergunta` colapsa as réplicas dentro de (condição × pergunta); o IC sai da variação
**entre perguntas**, que é a fonte de variância real deste desenho. Bootstrap percentílico
porque as métricas são binárias ou contagens pequenas — normal aproximada seria otimista.

In [ ]:
METRICAS = ['final_correct', 'corrected', 'esr', 'propagation_depth',
            'amplification', 'rcca', 'fesa', 'schema_validity',
            'contract_validity', 'pipeline_tokens', 'pipeline_cost_usd',
            'latency_ms', 'n_calls']

def por_pergunta(dados, metricas=METRICAS):
    """Uma linha por (condicao, pergunta): as replicas de seed viram uma media."""
    num = dados.copy()
    for m in metricas:
        num[m] = pd.to_numeric(num[m], errors='coerce')
    return num.groupby(['condition', 'question_id'])[metricas].mean().reset_index()

def ic_bootstrap(valores, n=5000, alpha=0.05, seed=0):
    """IC percentilico da media. None quando nao ha dado suficiente."""
    v = pd.Series(valores).dropna().to_numpy(dtype=float)
    if len(v) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    medias = rng.choice(v, size=(n, len(v)), replace=True).mean(axis=1)
    return tuple(np.quantile(medias, [alpha / 2, 1 - alpha / 2]))

def resumo(dados, metricas=METRICAS):
    """media, n e IC95% por condicao."""
    pp = por_pergunta(dados, metricas)
    linhas = []
    for cond, g in pp.groupby('condition'):
        for m in metricas:
            lo, hi = ic_bootstrap(g[m])
            linhas.append({'condition': cond, 'metrica': m, 'media': g[m].mean(),
                           'n_perguntas': int(g[m].notna().sum()),
                           'ic95_lo': lo, 'ic95_hi': hi})
    return pd.DataFrame(linhas)

principal = df[df.injection_stage == 'extractor']
subestudo = df[df.injection_stage == 'analyst']
print(f'estudo principal: {len(principal)} celulas | subestudo de estagio: {len(subestudo)} celulas')

## 3. Correctness — o resultado final está correto?

`final_correct` = Final Factual Accuracy; `corrected` = Correction Rate (`None` quando o
controle já errava ou quando o erro foi `sem_efeito`: sem isso a métrica infla).

In [ ]:
tab = resumo(principal, ['final_correct', 'corrected'])
display(tab.pivot(index='metrica', columns='condition', values='media').round(3))

# acuracia por tipo de erro (a proposta pede a decomposicao)
display(principal.pivot_table(index='error_type', columns='condition',
                              values='final_correct', aggfunc='mean').round(3))

## 4. Containment — até onde o erro contamina o pipeline?

In [ ]:
cont = resumo(principal, ['esr', 'propagation_depth', 'amplification'])
display(cont.pivot(index='metrica', columns='condition', values='media').round(3))

fig, eixos = plt.subplots(1, 3, figsize=(13, 3.4))
for eixo, m in zip(eixos, ['esr', 'propagation_depth', 'amplification']):
    d = cont[cont.metrica == m].set_index('condition').reindex(CONDICOES)
    erro = np.vstack([d.media - d.ic95_lo, d.ic95_hi - d.media])
    eixo.bar(d.index, d.media, yerr=erro, capsize=4, color='#4c72b0')
    eixo.set_title(m); eixo.set_xlabel('condicao')
fig.suptitle('Containment por condicao (IC95% bootstrap sobre perguntas)')
fig.tight_layout(); plt.show()

In [ ]:
# destino do erro -- passo 6 do protocolo
destino = (principal.pivot_table(index='fate', columns='condition',
                                 values='question_id', aggfunc='count')
                    .fillna(0).astype(int))
display(destino)
print('\nlembrete: sem_efeito NAO e contencao -- reporte como qualidade da amostra')
(destino / destino.sum()).round(3).T.plot(kind='barh', stacked=True, figsize=(9, 3))
plt.title('destino do erro por condicao'); plt.xlabel('fracao das celulas'); plt.show()

## 5. Root-Cause Observability — RCCA, FESA e a linha de base do auditor

RCCA/FESA só se interpretam contra a taxa de falso-positivo: um auditor viciado em sempre
culpar alguém pontuaria alto sem saber de nada. A base sai das células **limpas**.

In [ ]:
obs = resumo(df, ['rcca', 'fesa'])
display(obs.pivot(index='metrica', columns='condition', values='media').round(3))

fp = []
for c in controles:
    if c.audit:
        fp.append({'condition': str(c.condition),
                   'culpou_alguem': c.audit.get('first_error_stage') != 'none',
                   'confianca': c.audit.get('confidence')})
fp = pd.DataFrame(fp)
if len(fp):
    display(fp.groupby('condition').agg(falso_positivo=('culpou_alguem', 'mean'),
                                        confianca_media=('confianca', 'mean'),
                                        n=('culpou_alguem', 'size')).round(3))
else:
    print('sem celulas de controle auditadas -- rode o experimento com --include-clean')

## 6. Schema Validity e as checagens de contrato

`schema_validity` é ~1.0 por construção com structured outputs nativo (e `None` em A, que
não tem contrato). O sinal não-trivial é `contract_validity`: `derived_from` pendente,
`claim_id` duplicado, `generated_by` incoerente, evidência não-literal em D.

In [ ]:
val = resumo(df, ['schema_validity', 'contract_validity'])
display(val.pivot(index='metrica', columns='condition', values='media').round(3))

# lineage_agreement: sinal SECUNDARIO, valida o instrumento textual onde ha proveniencia
display(df.groupby('condition')['lineage_agreement'].mean().round(3))

## 7. Custo operacional

O custo do auditor fica **fora** do custo do pipeline (ele é instrumento de medida, não
estágio). O Extrator compartilhado é custo fixo comum às quatro condições e não aparece
aqui — o que se compara é o custo variável de Analista + Redator.

In [ ]:
custo = resumo(df, ['pipeline_tokens', 'pipeline_cost_usd', 'latency_ms', 'n_calls'])
display(custo.pivot(index='metrica', columns='condition', values='media').round(5))
print('custo total do auditor: US$ %.4f' % df.auditor_cost_usd.fillna(0).sum())
print('custo total do pipeline: US$ %.4f' % df.pipeline_cost_usd.fillna(0).sum())

## 8. As três RQs — comparações pareadas

RQ1 = A×B (estrutura), RQ2 = B×C (lineage), RQ3 = C×D (evidência). Pareado **por pergunta**:
as duas condições viram literalmente o mesmo estado corrompido, então a diferença dentro da
pergunta é a estimativa limpa do efeito do contrato.

In [ ]:
def comparar(dados, esquerda, direita, metricas=METRICAS):
    pp = por_pergunta(dados, metricas)
    a = pp[pp.condition == esquerda].set_index('question_id')[metricas]
    b = pp[pp.condition == direita].set_index('question_id')[metricas]
    comuns = a.index.intersection(b.index)
    dif = (b.loc[comuns] - a.loc[comuns])
    linhas = []
    for m in metricas:
        lo, hi = ic_bootstrap(dif[m])
        linhas.append({'metrica': m, f'{esquerda}': a.loc[comuns, m].mean(),
                       f'{direita}': b.loc[comuns, m].mean(),
                       'delta': dif[m].mean(), 'ic95_lo': lo, 'ic95_hi': hi,
                       'n_pares': int(dif[m].notna().sum())})
    return pd.DataFrame(linhas).round(4)

for rq, (e, d) in {'RQ1 estrutura (AxB)': ('A', 'B'),
                   'RQ2 lineage (BxC)': ('B', 'C'),
                   'RQ3 evidencia (CxD)': ('C', 'D')}.items():
    print('\n' + '=' * 70 + f'\n{rq}')
    display(comparar(principal, e, d,
                     ['final_correct', 'corrected', 'esr', 'propagation_depth',
                      'amplification', 'rcca', 'fesa', 'pipeline_tokens',
                      'pipeline_cost_usd', 'latency_ms']))

## 9. Subestudo de atribuição de estágio (injeção no Analista)

Único braço onde a classe `percentage` existe (o Extrator só extrai figuras brutas). Aqui o
Analista também roda compartilhado, então a Propagation Depth máxima é 1: o propósito é
**FESA**, não profundidade.

In [ ]:
if len(subestudo):
    display(resumo(subestudo, ['fesa', 'rcca', 'esr', 'final_correct'])
            .pivot(index='metrica', columns='condition', values='media').round(3))
    print('\nFESA por condicao, so nas celulas com erro percentual:')
    display(subestudo[subestudo.error_type == 'percentage']
            .pivot_table(index='error_type', columns='condition', values='fesa', aggfunc='mean').round(3))
else:
    print('subestudo ainda nao executado (run_experiment.py --injection-stage analyst)')

## 10. Tabela final para o artigo

Uma linha por condição, com IC95%. É a tabela que vai para a seção de resultados.

In [ ]:
linhas = []
for cond in CONDICOES:
    g = por_pergunta(principal)[lambda x: x.condition == cond]
    if not len(g):
        continue
    linha = {'condicao': cond, 'n_perguntas': g.question_id.nunique()}
    for m in ['final_correct', 'corrected', 'esr', 'propagation_depth',
              'amplification', 'rcca', 'fesa', 'schema_validity',
              'contract_validity', 'pipeline_tokens', 'pipeline_cost_usd']:
        lo, hi = ic_bootstrap(g[m])
        linha[m] = f'{g[m].mean():.3f} [{lo:.3f}, {hi:.3f}]' if pd.notna(g[m].mean()) else '-'
    linhas.append(linha)

tabela = pd.DataFrame(linhas).set_index('condicao')
display(tabela.T)
out_path = f'tabela_resultados_{MODEL_SLUG}.csv'
tabela.T.to_csv(out_path)
print(f'gravado em analysis/{out_path}')